# REST API — FastAPI (Optimized)

Run this cell to start the server, then send requests to `http://localhost:8000/predict`

In [ ]:
import joblib
import pandas as pd
import numpy as np
from pydantic import BaseModel
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
import uvicorn
import threading
from pathlib import Path

# Load once at startup — fast: model + scaler + column names (no dataset)
model = joblib.load('../models/churn_model.pkl')
scaler = joblib.load('../models/scaler.pkl')
feature_cols = joblib.load('../models/feature_cols.pkl')

# Predefined one-hot column templates (drop_first encoding)
# InternetService: reference=No, cols: InternetService_DSL, InternetService_Fiber optic
# Contract: reference=Month-to-month, cols: Contract_One year, Contract_Two year
# etc.

OH_MAP = {
    'MultipleLines': {'Yes': 'MultipleLines_Yes', 'No phone service': 'MultipleLines_No phone service'},
    'InternetService': {'Fiber optic': 'InternetService_Fiber optic', 'No': 'InternetService_No'},
    'OnlineSecurity': {'Yes': 'OnlineSecurity_Yes', 'No internet service': 'OnlineSecurity_No internet service'},
    'OnlineBackup': {'Yes': 'OnlineBackup_Yes', 'No internet service': 'OnlineBackup_No internet service'},
    'DeviceProtection': {'Yes': 'DeviceProtection_Yes', 'No internet service': 'DeviceProtection_No internet service'},
    'TechSupport': {'Yes': 'TechSupport_Yes', 'No internet service': 'TechSupport_No internet service'},
    'StreamingTV': {'Yes': 'StreamingTV_Yes', 'No internet service': 'StreamingTV_No internet service'},
    'StreamingMovies': {'Yes': 'StreamingMovies_Yes', 'No internet service': 'StreamingMovies_No internet service'},
    'Contract': {'One year': 'Contract_One year', 'Two year': 'Contract_Two year'},
    'PaymentMethod': {
        'Credit card (automatic)': 'PaymentMethod_Credit card (automatic)',
        'Electronic check': 'PaymentMethod_Electronic check',
        'Mailed check': 'PaymentMethod_Mailed check',
    },
}

BINARY_MAP = {'Yes': 1, 'No': 0}
NUM_COLS = ['tenure', 'MonthlyCharges', 'TotalCharges']
col_idx = {c: i for i, c in enumerate(feature_cols)}

app = FastAPI(title='Churn Prediction API', version='1.0.0')

In [ ]:
class CustomerInput(BaseModel):
    gender: str
    SeniorCitizen: int
    Partner: str
    Dependents: str
    tenure: int
    PhoneService: str
    MultipleLines: str
    InternetService: str
    OnlineSecurity: str
    OnlineBackup: str
    DeviceProtection: str
    TechSupport: str
    StreamingTV: str
    StreamingMovies: str
    Contract: str
    PaperlessBilling: str
    PaymentMethod: str
    MonthlyCharges: float
    TotalCharges: float

class PredictionOut(BaseModel):
    churn: int
    probability: float

In [ ]:
@app.post('/predict', response_model=PredictionOut)
def predict(data: CustomerInput):
    row = [0.0] * len(feature_cols)
    row[col_idx['gender_Male']] = 1.0 if data.gender == 'Male' else 0.0
    row[col_idx['SeniorCitizen']] = float(data.SeniorCitizen)
    row[col_idx['Partner']] = float(BINARY_MAP[data.Partner])
    row[col_idx['Dependents']] = float(BINARY_MAP[data.Dependents])
    row[col_idx['tenure']] = data.tenure
    row[col_idx['PhoneService']] = float(BINARY_MAP[data.PhoneService])
    row[col_idx['PaperlessBilling']] = float(BINARY_MAP[data.PaperlessBilling])
    row[col_idx['MonthlyCharges']] = data.MonthlyCharges
    row[col_idx['TotalCharges']] = data.TotalCharges
    for raw_col, mapping in OH_MAP.items():
        val = getattr(data, raw_col)
        if val in mapping:
            row[col_idx[mapping[val]]] = 1.0
    row_df = pd.DataFrame([row], columns=feature_cols)
    row_df[NUM_COLS] = scaler.transform(row_df[NUM_COLS])
    prob = float(model.predict_proba(row_df)[0, 1])
    return PredictionOut(churn=int(prob >= 0.5), probability=round(prob, 4))

In [ ]:
@app.get('/health')
def health():
    return {'status': 'ok'}

### Start server

In [ ]:
threading.Thread(target=uvicorn.run, args=(app,), kwargs={'host': '0.0.0.0', 'port': 8000, 'log_level': 'error'}, daemon=True).start()
print('Server started at http://localhost:8000')
print('API docs at http://localhost:8000/docs')

### Test prediction

In [ ]:
import httpx

sample = {
    'gender': 'Male', 'SeniorCitizen': 0, 'Partner': 'Yes', 'Dependents': 'No',
    'tenure': 12, 'PhoneService': 'Yes', 'MultipleLines': 'No',
    'InternetService': 'Fiber optic', 'OnlineSecurity': 'No', 'OnlineBackup': 'Yes',
    'DeviceProtection': 'No', 'TechSupport': 'No', 'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes', 'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check', 'MonthlyCharges': 70.0, 'TotalCharges': 840.0,
}

resp = httpx.post('http://localhost:8000/predict', json=sample, timeout=10)
print(resp.status_code, resp.json())